## Appendix: Rubric mapping (for TAs / instructors)

| Rubric section | Where it is covered in *this* notebook |
|----------------|----------------------------------------|
| 1. Problem Framing | Opening **## 1. Problem Framing** markdown, then **Phase 1** code |
| 2. Data Acquisition, Preparation & Exploration | **## 2. Data Acquisition...** + **Phase 2** prep; **Phase 3** exploration (EDA / correlations) |
| 3. Modeling & Feature Selection | **## 3. Modeling & Feature Selection** + **Phase 4** modeling code |
| 4. Evaluation & Interpretation | **## 4. Evaluation & Interpretation** + **Phase 5** evaluation code |
| 5. Causal and Relationship Analysis | **## 5. Causal...** + **Phase 6** feature / impact code |
| 6. Deployment Notes | **## 6. Deployment Notes** + **Phase 7** artifact export |

Graded narrative is in the **## 1–6** cells placed next to their matching phases; this table is navigation only.


# Public impact snapshots — accountability & storytelling pipeline

## 1. Problem Framing
**What we solve:** The foundation must communicate **transparent, period-based impact** (services, beneficiary reach, and outcome-style indicators) to donors, partners, and leadership—aligned with an “annual accomplishment” style report. This notebook builds the **structured snapshot payload** that backs that narrative.

**Who cares:** executive leadership, communications, grant reporting, and the public **Impact** page audience.

**Predictive vs explanatory:** The primary deliverable is **descriptive + lightly model-assisted summarization** of indicators (not individual-level prediction). Where models appear, they support **aggregation and emphasis** (what to highlight), while avoiding over-claiming causal impact.


In [1]:
# Phase 1: Problem Framing
business_question = "What drives monthly donations, and how can we predict next month donations?"
success_metrics = ["Lower MAE", "Lower RMSE", "Higher R2 on test data"]
approach = {
    "causal_model": "Explanatory OLS regression (relationship estimates, not proof of causality)",
    "predictive_model": "Random Forest regression (best out-of-sample prediction)"
}

print("Business Question:", business_question)
print("Success Metrics:", success_metrics)
print("Approach:", approach)

Business Question: What drives monthly donations, and how can we predict next month donations?
Success Metrics: ['Lower MAE', 'Lower RMSE', 'Higher R2 on test data']
Approach: {'causal_model': 'Explanatory OLS regression (relationship estimates, not proof of causality)', 'predictive_model': 'Random Forest regression (best out-of-sample prediction)'}


## 2. Data Acquisition, Preparation & Exploration

### Source and schema
This pipeline starts from **`datasets/public_impact_snapshots.csv`**. Each row represents a **published impact snapshot** for a period (via `snapshot_date`), with a nested **`metric_payload_json`** field that stores the indicator bundle as JSON text. In a reporting product, that payload is what ultimately feeds **year-over-year copy**, bullets on the public **Impact** page, and leadership review—so parsing it cleanly is part of **data acquisition**, not an afterthought.

### Preparation choices (Phase 2 code)
- **`metric_payload_json`** is expanded with `ast.literal_eval` into typed columns so we can analyze **average health** (`avg_health_score`), **average education progress** (`avg_education_progress`), **resident counts** (`total_residents`), and the **donation total anchored to that month** (`donations_total_for_month`). These fields are the subset used in this notebook’s modeling slice; the full JSON may contain additional indicators elsewhere in the app.
- **Dates:** `snapshot_date` and `published_at` are parsed to datetimes. **`month_number`** and **`year`** are derived from `snapshot_date` so seasonality and calendar alignment are explicit features.
- **Missing values:** Rows with missing values in any modeling column are dropped for the analysis frame (`df_model`). That is a deliberate **complete-case** choice for this teaching notebook; a production pipeline might impute or model missingness explicitly and document the rule in the same place grants ask for “methodology.”

### Why `donations_total_for_month` appears here
Public impact storytelling and **fundraising reality** are linked in operations: leadership often looks at **program indicators** alongside **donation inflows** when deciding what to emphasize externally. This notebook uses monthly donation totals as an **outcome variable for exploration** to practice dual-track modeling (interpretable linear associations + a random forest for prediction). It does **not** claim that impact metrics *cause* donations; see **Section 5** for limits.

### Reproducibility and auditability
Anyone rerunning the notebook should get the same **row counts**, **feature matrix**, and **saved artifacts** (Phase 7). That supports the TA requirement that numbers in a slide deck or Impact page could, in principle, be traced back to committed data and code.


In [2]:
# Phase 2: Data Acquisition and Preparation
import ast
import json
import pandas as pd

file_path = "../datasets/public_impact_snapshots.csv"
df = pd.read_csv(file_path)

payload = df["metric_payload_json"].apply(ast.literal_eval).apply(pd.Series)
df2 = pd.concat([df.drop(columns=["metric_payload_json"]), payload], axis=1)

df2["snapshot_date"] = pd.to_datetime(df2["snapshot_date"])
df2["published_at"] = pd.to_datetime(df2["published_at"], errors="coerce")
df2["month_number"] = df2["snapshot_date"].dt.month
df2["year"] = df2["snapshot_date"].dt.year

df_model = df2[[
    "avg_health_score",
    "avg_education_progress",
    "total_residents",
    "month_number",
    "year",
    "donations_total_for_month"
]].dropna()

print("Rows and Columns:", df_model.shape)
print(df_model.head())

Rows and Columns: (50, 6)
   avg_health_score  avg_education_progress  total_residents  month_number  \
0              3.03                   33.90               60             1   
1              3.13                   51.05               60             2   
2              3.16                   60.57               60             3   
3              3.20                   69.15               60             4   
4              3.20                   78.85               60             5   

   year  donations_total_for_month  
0  2023                    1379.92  
1  2023                    2065.15  
2  2023                    9577.83  
3  2023                    5401.47  
4  2023                    4862.09  


## 2 (continued). Exploration (EDA)

### What we look for before modeling
Phase 3 is intentionally lightweight (tables, not a full visualization suite) because this notebook’s emphasis is **measurement hygiene + honest interpretation** for a public-facing storyline. Still, we ask the same questions an analyst would ask in Chapter 6–8 of the course text:

- **Scale and skew:** Do outcomes and inputs live on sensible ranges? Do a few extreme months dominate totals?
- **Missingness and constants:** Are any predictors effectively fixed (e.g., the same population denominator every month)? Constants break correlation and can destabilize regression.
- **Bivariate structure:** Which indicators move **together** with `donations_total_for_month` in simple correlations?

### How to read the correlation block
The code prints Pearson correlations with **`donations_total_for_month`**. In the sample output you will typically see **positive** associations for **`avg_education_progress`** and **`avg_health_score`**, a **seasonal** pattern with **`month_number`**, and sometimes a **negative** correlation with **`year`** depending on how campaigns and reporting windows line up—those are **historical co-movements**, not proof that improving health scores “produces” donations.

### Limits of this EDA slice
- **Small \(n\):** Dozens of monthly rows are enough for a classroom pipeline but not for high-precision inference.
- **No individual-level outcomes:** Everything is aggregated; you cannot attribute donations to a single beneficiary story.
- **`total_residents`:** If this column is constant in the extract, correlations involving it may be undefined (**NaN**). That is a signal to **drop or reshape** the feature for reporting, not to force it into interpretation.

The Phase 3 cell below prints `describe()` and the correlation vector so you can connect numbers to the narrative in Sections 3–5.


In [3]:
# Phase 3: Exploration
print("Summary Statistics")
print(df_model.describe())

print("\nCorrelations with donations_total_for_month")
print(df_model.corr(numeric_only=True)["donations_total_for_month"].sort_values(ascending=False))

Summary Statistics
       avg_health_score  avg_education_progress  total_residents  \
count          50.00000               50.000000             50.0   
mean            2.45060               59.499000             60.0   
std             1.40025               35.194204              0.0   
min             0.00000                0.000000             60.0   
25%             3.04250               38.187500             60.0   
50%             3.13500               77.485000             60.0   
75%             3.20000               81.717500             60.0   
max             3.94000              100.000000             60.0   

       month_number         year  donations_total_for_month  
count     50.000000    50.000000                  50.000000  
mean       6.300000  2024.600000                4814.490800  
std        3.558548     1.212183                4512.487236  
min        1.000000  2023.000000                   0.000000  
25%        3.000000  2024.000000                 902.43750

## 3. Modeling & Feature Selection (where applicable)

### Two tracks, two audiences
We fit **two models on the same feature matrix** (Phase 4), mirroring the textbook split between **explanation** and **prediction**:

1. **OLS (via statsmodels)** — An **explanatory / associational** linear model. Coefficients are easy to narrate to non-technical stakeholders (“holding other included variables fixed in this specification, a one-unit change in X is associated with …”). This is **not** a causal identification strategy.
2. **Random forest in a scaled pipeline** — A **predictive** model tuned for out-of-sample error. It can capture **non-linear** and **interaction** structure (e.g., month effects combining differently with education progress) that OLS may miss.

### Feature set and pillar mapping
Predictors are chosen because they align with program pillars we already communicate publicly:

| Feature | Pillar / meaning |
|--------|-------------------|
| `avg_health_score` | **Healing** — population-level wellbeing signal in the snapshot period |
| `avg_education_progress` | **Teaching** — learning / progression signal captured in the payload |
| `total_residents` | **Reach / scale** — denominator-style load (when it varies) |
| `month_number`, `year` | **Calendar context** — seasonality, reporting cycles, and slow trends |

We deliberately keep the set **small and legible** for dashboards and grant footnotes. A different project might add safehouse-level fixed effects, campaign dummies, or macro controls—only if the data support and the story need them.

### Train / test split caveat
The code uses a **random** `train_test_split`. For real monthly fundraising data you would often prefer **time-based** validation (train on past months, test on future months) to respect autocorrelation. We note the gap here so you can defend a stronger evaluation design in writeups without pretending this notebook is the final word on temporal generalization.

### What we are not doing yet
Feature selection beyond “domain + sanity checks” happens **after** you inspect stability (Section 4) and interpret coefficients and importances together (Section 5). We avoid cherry-picking only the features that make the public story look best.


In [4]:
# Phase 4: Modeling (Causal + Predictive)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm

X = df_model.drop(columns=["donations_total_for_month"])
y = df_model["donations_total_for_month"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Causal / explanatory model (OLS)
X_train_ols = sm.add_constant(X_train)
causal_model = sm.OLS(y_train, X_train_ols).fit()

# Predictive model
predictive_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])
predictive_model.fit(X_train, y_train)

print("Causal model and predictive model trained.")

Causal model and predictive model trained.


## 4. Evaluation & Interpretation

### Operational success vs textbook metrics
For Lighthouse’s **Impact** narrative, “good” analytics means at least three things:

1. **Auditability** — Published percentages and averages can be tied to **this CSV**, the parsed JSON fields, and the notebook outputs (same row counts, same definitions).
2. **Stability** — Month-to-month indicator movements are **plausible** given what program staff observe; huge swings trigger a data-quality review, not automatic cheerleading.
3. **Fit (when models are used)** — If leadership uses the random forest for **rough planning** (“directionally, what donation range goes with these indicators?”), then **MAE / RMSE / R²** on holdout data matter—but always second to honesty about uncertainty.

### Reading the holdout metrics (Phase 5)
- **MAE (mean absolute error)** in the same units as monthly donations: interpret as **typical** peso-level miss if you treat the model as a point forecast.
- **RMSE** penalizes large errors more heavily—useful if big misses are especially costly for staffing or campaign timing.
- **R²** summarizes incremental variance explained **relative to the mean predictor** on the test split. For public communications, prefer language like “explains a modest share of historical variation” unless your validation design and sample size justify stronger claims.

### OLS output in the same phase
The cell also prints the **OLS summary**. Here:

- **R² on the training portion** reflects **in-sample** fit for that linear specification; it is **not** directly comparable to the forest’s test R² without aligned procedures.
- **p-values** are only meaningful under the usual regression assumptions (linearity, exogeneity, homoskedasticity, etc.), which we have **not** fully validated—treat stars as **descriptive** cues, not rigid hypothesis tests.
- The summary may warn about a **large condition number**, hinting at **multicollinearity** (e.g., calendar features correlated with trend). That is a prompt to temper causal language and to consider regularization or re-specification in a production analysis.

### Fairness and gaps
Phase 5 notes that **demographic group columns** are absent from this extract. That avoids sloppy fairness charts, but it also means **equity questions** must be addressed in other pipelines (`residents`, `education_records`, etc.) where protected attributes may appear—**this notebook should not be read as the organization’s full fairness posture.**

### Closing the loop with decisions
Evaluation is incomplete until someone states **how errors hurt decisions**: overestimating donations can over-commit programs; underestimating can starve services. Tie metrics to those asymmetric costs when you write recommendations (Section 5–6).


In [5]:
# Phase 5: Evaluation and Selection
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = predictive_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Predictive Model Metrics")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 4))

# Fairness check (simple): no demographic group columns available
print("Fairness check: demographic group fields are not present in this dataset.")

print("\nCausal Model (OLS) summary:")
print(causal_model.summary())

Predictive Model Metrics
MAE: 2062.78
RMSE: 2709.36
R2: 0.3895
Fairness check: demographic group fields are not present in this dataset.

Causal Model (OLS) summary:
                                OLS Regression Results                               
Dep. Variable:     donations_total_for_month   R-squared:                       0.584
Model:                                   OLS   Adj. R-squared:                  0.537
Method:                        Least Squares   F-statistic:                     12.30
Date:                       Tue, 07 Apr 2026   Prob (F-statistic):           2.40e-06
Time:                               22:05:18   Log-Likelihood:                -376.99
No. Observations:                         40   AIC:                             764.0
Df Residuals:                             35   BIC:                             772.4
Df Model:                                  4                                         
Covariance Type:                   nonrobust                

## 5. Causal and Relationship Analysis

### Language we allow (and language we forbid)
This section is the **integrity checkpoint** for a notebook that mixes **public storytelling** with **donation outcomes**.

- **Allowed:** “In our historical snapshots, months with higher recorded **education progress** and **health scores** tend to coincide with higher **donation totals**, after accounting for the other variables in this simple specification.”
- **Allowed:** “The **random forest** places more weight on **calendar / seasonality** features than the linear model, which reminds us that fundraising has strong **timing** effects.”
- **Forbidden:** “Improving health scores **causes** donors to give more.”
- **Forbidden:** “This coefficient proves that our teaching programs increased revenue.”

**Causal inference** requires a design: randomization, credible instruments, differences-in-differences with parallel trends, regression discontinuity, or other strategies that defend **exogeneity**. We have **observational monthly aggregates** and a handful of indicators—enough for **association**, monitoring, and **hypothesis generation**, not for definitive causal claims.

### What the OLS block is actually estimating
The Phase 4–5 **OLS** model relates `donations_total_for_month` to **`avg_health_score`**, **`avg_education_progress`**, **`total_residents`**, **`month_number`**, and **`year`**. Interpret each coefficient as a **conditional association** inside this textbook specification:

- **Sign and magnitude** answer: “If this linear approximation were trustworthy, how much of the donation **level** co-moves with one more unit of X, **given** the other included regressors?”
- They do **not** answer: “What happens if we intervene to change X in the real world?” Interventions shift **multiple** margins at once (communications, staffing, emergencies, donor lists) that are not fully captured here.

When the printed summary flags a **high condition number**, read that as **multicollinearity / scaling sensitivity**: calendar features (`month_number`, `year`) often **overlap** in short panels. Inflated standard errors and “flipping” signs versus simple correlations are **not rare**—they mean linear decomposition is **unstable**, not that donors dislike education.

### Why coefficients and correlations can disagree
Phase 3 showed **raw** correlations with donations. Phase 6 shows **OLS coefficients** (after adding controls) and **forest importances** (nonlinear weighting). It is normal for:

- **Education progress** to look **positively** correlated with donations but **negatively** or **insignificant** in OLS if **seasonality** or **year** soaks up shared variance (suppression / confounding structure).
- **Health** to appear **positive** in both correlation and OLS while education looks weaker—perhaps because both ride on a **latent “good months”** factor we do not observe (fundraising pushes, external events, recording quality).

The correct response is **transparent reporting**: show the bivariate picture **and** the multivariate picture, then explain that **different slices answer different questions**.

### Random forest “importance” is not causal priority
Phase 6 prints **feature importances** from sklearn’s forest. They summarize how much the ensemble relies on each column for **prediction** under its default split rules. They can be useful for **monitoring** and for deciding what to **track** on dashboards, but:

- Importances can be **skewed** when predictors are correlated.
- They **absorb nonlinearities** (e.g., December vs June) that OLS spreads across multiple terms.
- High importance **does not** rank ethical or strategic “what should we do next” without domain judgment.

Use them to **complement**, not replace, program theory and staff context.

### Confounders you should name in writeups
Before any “drivers” language sneaks into donor-facing copy, list plausible **omitted variables** that are missing from `df_model` but move both programs and donations:

- **Fundraising intensity** (campaigns, grants, major gifts)
- **Macroeconomics** (inflation, disasters, holiday giving)
- **Earned media / crises**
- **Data recording** (late snapshots, corrected payloads)

If those drive both pillars and donations, coefficients are **biased** for causal purposes—even if they still help **describe historical co-movement.**

### How to phrase recommendations (Phase 6) responsibly
The printout’s bullet recommendations are **illustrative**. In your course submission, rewrite them as **conditional monitoring rules**, e.g.:

- “When **health** and **education** indicators deteriorate together, **investigate operations first**, then review fundraising plans—because confounding cannot be ruled out.”
- “Treat **seasonal** coefficients as **planning baselines**, not moral judgments about programs.”

### Connection to organizational ethics
Over-claiming causal impact erodes trust with **donors**, **partners**, and **beneficiaries**. This notebook deliberately keeps the math modest so the **narrative team** can say: our numbers are **consistent with** program strength and donor engagement, while uncertainty is **explicit**.

The Phase 6 cell below lists **OLS coefficients** and **forest importances** side by side—use it as the evidence panel for this section, then **qualify every sentence** that sounds like impact proof.


In [6]:
# Phase 6: Feature Selection / Impact
# Explanatory impact (OLS coefficients)
coef_table = causal_model.params.sort_values(key=lambda s: s.abs(), ascending=False)
print("Top explanatory coefficients (absolute size):")
print(coef_table)

# Predictive impact (Random Forest feature importances)
rf_model = predictive_model.named_steps["model"]
importance_table = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop predictive feature importances:")
print(importance_table)

print("\nRecommended decisions based on top features:")
print("1) Prioritize programs that improve education progress.")
print("2) Track health score trends monthly and intervene early if scores drop.")
print("3) Use seasonal month patterns for campaign timing.")

Top explanatory coefficients (absolute size):
total_residents          -19246.983147
avg_health_score           4083.583650
month_number                728.832332
year                        567.528788
avg_education_progress      -67.336047
dtype: float64

Top predictive feature importances:
month_number              0.376806
avg_education_progress    0.317571
avg_health_score          0.170878
year                      0.134745
total_residents           0.000000
dtype: float64

Recommended decisions based on top features:
1) Prioritize programs that improve education progress.
2) Track health score trends monthly and intervene early if scores drop.
3) Use seasonal month patterns for campaign timing.


## 6. Deployment Notes

### What ships to production vs what lives in `ml-pipelines`
- **This notebook** (under `ml-pipelines/`) is the **authoring** environment: parse the snapshot CSV, explore relationships, train models, and emit **versioned artifacts**.
- The **ML service** serves a **JSON cache** that the web tier can read quickly without re-running sklearn on every request. Paths differ by repo layout; keep **one canonical** `impact_pipeline_cache.json` (commonly under `ml-service/artifacts/`) so operators know what the API returned last.

### API surface (read-only analytics)
**`GET /impact/analytics`** in `ml-service/app/main.py` returns **`impact_pipeline_cache.json`** (or a documented sample fallback when cache population is skipped in dev). Downstream callers should treat the payload as **pre-aggregated public metrics**, not as personally identifiable data.

### Frontend consumption
The Lighthouse **Impact** experience pulls these summaries through the normal **backend → frontend** chain: the UI should **never** hard-code notebook outputs—but it **should** display the same definitions (what “average education progress” means, which month window, currency for donations) that this pipeline documents.

### Artifacts Phase 7 produces
Phase 7 writes **at least**:

- **`public_impact_snapshots_production_model.joblib`** — serialized sklearn **pipeline** (scaler + random forest) for offline scoring or service-side loads if you choose to invoke it.  
- **`public_impact_snapshots_sample_payload.json`** — a **contract example** showing the feature keys expected at scoring time.

When you change features in Phase 2–4, you must **re-run to completion**, commit refreshed artifacts, and bump any **schema/version** notes consumed by `main.py` or deployment docs—otherwise the Impact page and the notebook will silently diverge.

### Operations checklist
1. **Refresh cadence:** align retrains with **snapshot publication** (e.g., monthly after finance closes donation totals).  
2. **Validation:** compare a handful of API JSON fields to **`df_model` aggregates** after each release.  
3. **Failure mode:** if the model file is missing, fall back to **descriptive-only** analytics rather than guessing predictions—Section 5’s cautions still apply.  
4. **Governance:** causal language in marketing copy should be reviewed with this section’s constraints, not only with model metrics.


In [7]:
# Phase 7: Deployment (simple artifact + sample payload)
import os
import joblib

os.makedirs("../artifacts", exist_ok=True)

artifact_path = "../artifacts/public_impact_snapshots_production_model.joblib"
joblib.dump(predictive_model, artifact_path)

sample_payload = {
    "avg_health_score": 3.2,
    "avg_education_progress": 78.0,
    "total_residents": 60,
    "month_number": 4,
    "year": 2026
}

sample_df = pd.DataFrame([sample_payload])
sample_prediction = predictive_model.predict(sample_df)[0]

with open("../artifacts/public_impact_snapshots_sample_payload.json", "w", encoding="utf-8") as f:
    json.dump(sample_payload, f, indent=2)

print("Saved model to:", artifact_path)
print("Sample predicted monthly donations:", round(float(sample_prediction), 2))

Saved model to: ../artifacts/public_impact_snapshots_production_model.joblib
Sample predicted monthly donations: 4563.41
